<a href="https://colab.research.google.com/github/AvantiShri/gotpsi_sequentialcard/blob/main/notebooks/GotPsi_SequentialCard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get the data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/IONS/Dean_SequentialCard

 cardS16.zip  'Dean SequentialCard README.gdoc'  'raw cardsequence.zip'


In [3]:
!md5sum "/content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip" #md5 checksum to compare integrity of file

29899584767d64ec6d3cfcd7496d751a  /content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip


In [4]:
!unzip -q "/content/drive/MyDrive/IONS/Dean_SequentialCard/cardS16.zip" -d /content/cardsequence #takes about a minute

In [5]:
#The data for the first two quarters of 2002 is hiding in cardS02Q1.zip and cardS02Q2.zip, so we will further
# unzip those
!unzip -q /content/cardsequence/cardS02/cardS02Q1.zip -d /content/cardsequence/cardS02
!unzip -q /content/cardsequence/cardS02/cardS02Q2.zip -d /content/cardsequence/cardS02

# Install dependencies

In [7]:
! pip uninstall -y gotpsi_sequentialcard
! pip install git+https://github.com/AvantiShri/gotpsi_sequentialcard.git

  Cloning https://github.com/AvantiShri/gotpsi_sequentialcard.git to /tmp/pip-req-build-z4gki3dp
  Running command git clone --filter=blob:none --quiet https://github.com/AvantiShri/gotpsi_sequentialcard.git /tmp/pip-req-build-z4gki3dp
  Resolved https://github.com/AvantiShri/gotpsi_sequentialcard.git to commit 13026506d9ef3f33c13b22d6b4cfc8eb8b3909de
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gotpsi_sequentialcard: filename=gotpsi_sequentialcard-0.1.0-py3-none-any.whl size=2822 sha256=ea3a1db46843236c77b22bb812a49511e945fe49069305eabb2f30294adb8dcd
  Stored in directory: /tmp/pip-ephem-wheel-cache-zzxni0t8/wheels/da/7d/d8/9daffca3dce198e9fd8b21af1736b50aef399566987d254017
Successfully built gotpsi_sequentialcard


In [8]:
from importlib import reload
import gotpsi_sequentialcard.utils
reload(gotpsi_sequentialcard.utils)

<module 'gotpsi_sequentialcard.utils' from '/usr/local/lib/python3.12/dist-packages/gotpsi_sequentialcard/utils.py'>

In [9]:
from gotpsi_sequentialcard.utils import find_missing_dates

missing_ranges, earliest_date, latest_date, available_dates = find_missing_dates("/content/cardsequence")

print(f"Data spans: {earliest_date} to {latest_date}")
print(f"\n{len(missing_ranges)} missing date range(s):\n")

for start, end in missing_ranges:
    if start == end:
        print(f"  {start}")
    else:
        days = (end - start).days + 1
        print(f"  {start} to {end}  ({days} days)")

Data spans: 2002-01-01 to 2018-12-29

6 missing date range(s):

  2015-08-13 to 2015-09-20  (39 days)
  2015-09-24 to 2015-09-27  (4 days)
  2015-09-29 to 2015-10-03  (5 days)
  2015-10-07 to 2015-10-09  (3 days)
  2015-10-11 to 2015-10-26  (16 days)
  2015-12-04 to 2016-05-12  (161 days)


In [10]:
from collections import namedtuple
from datetime import datetime
import os

Trial = namedtuple('Trial', ['userid', 'trial', 'target_card', 'click_sequence', 'image', 'timestamp'])

# Completed trial format:
#   userid, trial, steps, r1,r2,r3,r4,r5,r6, imagepath, timestamp
# Example: rada, 1, 2, 0,5,4,0,0,0, ../../bi/images4/c9.jpg, Thu Jan  1 00:05:52 2009
def parse_trial_file(filepath):
    trials = []
    error_lines = []

    with open(filepath, 'r', encoding='latin-1') as f:
        try:
            for line_num, line in enumerate(f, 1):
                if '/images' not in line:
                    continue

                parts = [p.strip() for p in line.split(', ')]
                assert len(parts) == 6, f"Expected 6 parts, got {len(parts)}"

                userid = parts[0]
                if (parts[1]==""):
                    raise RuntimeError("Missing trial number")
                trial_num = int(parts[1])
                steps = int(parts[2])
                clicks = parts[3].split(",")
                imagepath = parts[4]
                timestamp_str = parts[5].strip()
                timestamp = datetime.strptime(timestamp_str, '%a %b %d %H:%M:%S %Y')

                image = os.path.basename(imagepath)

                assert clicks[0] == "0", (
                    f"Expected r1=0, got {clicks[0]}"
                )

                trailing = clicks[steps + 1:]
                assert len(trailing) == 0 or all(r == "0" for r in trailing), (
                    f"Expected trailing zeros after step {steps}, got {clicks}"
                )

                click_sequence = [int(x) for x in clicks[1:steps + 1] if x != '']
                if (len(click_sequence) < steps):
                  print(f"Skipped empty entries in click sequence for {line.rstrip()}")
                assert len(set(click_sequence))==len(click_sequence), (
                    f"Expected unique click sequence, got {click_sequence}")
                target_card = click_sequence[-1]

                trials.append(Trial(userid, trial_num, target_card, click_sequence, image, timestamp))

        except Exception as e:
            print(f"Error in {filepath}, line {line_num}: {e}")
            print(f"  Content: {line.rstrip()}")
            error_lines.append(line)

    return trials, error_lines

In [11]:
from tqdm import tqdm

#read in data for all the dates
# Takes about 10 minutes and just over 14G of RAM - need to switch to a high RAM runtime
date_to_trials = {}

all_error_lines = []
for available_date in tqdm(sorted(available_dates)):
  #the filepath is /content/cardsequence/cardSYY/cardSYYMMDD.dat
  dat_filepath = f"/content/cardsequence/cardS{available_date.strftime('%y')}/cardS{available_date.strftime('%y')}{available_date.strftime('%m')}{available_date.strftime('%d')}.dat"
  if ("020129" in dat_filepath) or ("020130" in dat_filepath) or ("020131" in dat_filepath):
    #dat_filepath = dat_filepath.replace("020129", "020129a")
    print(f"Skipping {available_date} because of confusion regarding which file to use")
    continue
  trials, error_lines = parse_trial_file(dat_filepath)
  date_to_trials[available_date] = trials
  all_error_lines.extend(error_lines)

  1%|          | 34/5979 [00:00<00:24, 243.57it/s]

Skipping 2002-01-29 because of confusion regarding which file to use
Skipping 2002-01-30 because of confusion regarding which file to use
Skipping 2002-01-31 because of confusion regarding which file to use
Skipped empty entries in click sequence for jsbachrh, 10, 6, 0,4,1,,2,3,5, ../../bi/images4/c10.jpg, Fri Feb  1 04:31:31 2002
Skipped empty entries in click sequence for elsa, 13, 3, 0,,4,5,0,0, ../../bi/images4/c9.jpg, Fri Feb  1 14:29:29 2002
Error in /content/cardsequence/cardS02/cardS020201.dat, line 10645: Missing trial number
  Content: joy, , 3, 0,1,5,3,0,0, ../../bi/images4/c5.jpg, Fri Feb  1 21:11:21 2002
Skipped empty entries in click sequence for shadow drakken, 15, 5, 0,4,3,,5,1, ../../bi/images4/ok.jpg, Sat Feb  2 00:28:03 2002
Skipped empty entries in click sequence for shadow drakken, 17, 5, 0,4,2,3,,5, ../../bi/images4/c8.jpg, Sat Feb  2 00:28:30 2002
Skipped empty entries in click sequence for igryphon, 1, 6, 0,,2,4,1,5,3, ../../bi/images4/ok.jpg, Sat Feb  2 04:25:5

  1%|          | 59/5979 [00:01<02:42, 36.43it/s] 

Error in /content/cardsequence/cardS02/cardS020301.dat, line 2125: Missing trial number
  Content: oriok, , 4, 0,3,2,4,1,0, ../../bi/images4/c5.jpg, Fri Mar  1 04:00:13 2002
Skipped empty entries in click sequence for mafioso, 1, 5, 0,1,,5,2,4, ../../bi/images4/c5.gif, Sat Mar  2 02:44:36 2002
Skipped empty entries in click sequence for ng.andrew@mbox.com.au, 18, 6, 0,2,,4,5,3,1, ../../bi/images4/c3.gif, Sat Mar  2 03:38:15 2002
Skipped empty entries in click sequence for wildjoker8080, 13, 3, 0,,2,3,0,0, ../../bi/images4/c10.jpg, Sat Mar  2 06:58:47 2002
Skipped empty entries in click sequence for symple, 3, 5, 0,3,2,,5,4, ../../bi/images4/c3.jpg, Sat Mar  2 09:04:33 2002
Skipped empty entries in click sequence for nic finn, 23, 5, 0,,2,1,3,4, ../../bi/images4/c10.jpg, Sat Mar  2 10:51:20 2002
Skipped empty entries in click sequence for nenya, 12, 3, 0,,3,2,0,0, ../../bi/images4/c4.gif, Sat Mar  2 11:46:29 2002
Skipped empty entries in click sequence for blindfold, 22, 4, 0,,2,3,1,0, 

  1%|          | 71/5979 [00:01<03:01, 32.51it/s]

...

  1%|▏         | 79/5979 [00:02<03:09, 31.14it/s]

Skipped empty entries in click sequence for nic, 1, 3, 0,,3,1,0,0, ../../bi/images4/c7.gif, Sun Mar 17 03:16:21 2002
Error in /content/cardsequence/cardS02/cardS020317.dat, line 5958: Missing trial number
  Content: whoknowshow42, , 4, 0,4,3,1,5,0, ../../bi/images4/ok.jpg, Sun Mar 17 10:02:28 2002
Skipped empty entries in click sequence for simtusie, 22, 5, 0,,3,4,1,2, ../../bi/images4/c2.gif, Mon Mar 18 11:08:00 2002
Skipped empty entries in click sequence for jasmineg, 13, 5, 0,4,,2,3,5, ../../bi/images4/c5.jpg, Mon Mar 18 11:31:58 2002
Skipped empty entries in click sequence for gerardwood, 8, 6, 0,2,3,,5,4,1, ../../bi/images4/c1.gif, Mon Mar 18 12:51:45 2002
Skipped empty entries in click sequence for alse, 8, 5, 0,4,1,5,,2, ../../bi/images4/c8.gif, Mon Mar 18 17:09:06 2002
Error in /content/cardsequence/cardS02/cardS020318.dat, line 19952: Missing trial number
  Content: susan0138, , 3, 0,4,3,2,0,0, ../../bi/images4/c8.gif, Mon Mar 18 17:55:43 2002
Skipped empty entries in click s

 97%|█████████▋| 5790/5979 [08:10<00:04, 40.64it/s]

Error in /content/cardsequence/cardS18/cardS180614.dat, line 8939: Missing trial number
  Content: lweltzer, , 0, ,2,0,0,0,0,0, ../images4/c3.jpg, Thu Jun 14 19:50:55 2018
Error in /content/cardsequence/cardS18/cardS180615.dat, line 672: Missing trial number
  Content: conan01, , 0, ,3,0,0,0,0,0, ../images4/c4.gif, Fri Jun 15 06:41:00 2018


100%|██████████| 5979/5979 [08:14<00:00, 12.10it/s]


In [12]:
len(date_to_trials)

5976

In [16]:
len(all_error_lines)

47

In [13]:
sum([len(trials) for trials in date_to_trials.values()])

36863593

In [82]:
!cat /content/cardsequence/cardS02/cardS020111.dat | grep 'whisper11'

whisper11, 1, 1, Fri Jan 11 12:11:49 2002
whisper11, 1, 3, Fri Jan 11 12:12:02 2002
whisper11, 1, 2, Fri Jan 11 12:12:04 2002
whisper11, 1, 3, 0,1,3,2,0,0, images/c4.jpg, Fri Jan 11 12:12:04 2002
whisper11, 2, 3, Fri Jan 11 12:12:22 2002
whisper11, 2, 2, Fri Jan 11 12:12:24 2002
whisper11, 2, 1, Fri Jan 11 12:12:29 2002
whisper11, 2, 4, Fri Jan 11 12:12:32 2002
whisper11, 2, 5, Fri Jan 11 12:12:35 2002
whisper11, 2, 5, 0,3,2,1,4,5, images/c2.jpg, Fri Jan 11 12:12:35 2002
whisper11, 3, 4, Fri Jan 11 12:12:49 2002
whisper11, 3, 5, Fri Jan 11 12:12:54 2002
whisper11, 3, 3, Fri Jan 11 12:12:57 2002
whisper11, 3, 2, Fri Jan 11 12:13:01 2002
whisper11, 3, 4, 0,4,5,3,2,0, images/ok.jpg, Fri Jan 11 12:13:01 2002
whisper11, 4, 4, Fri Jan 11 12:13:15 2002
whisper11, 4, 3, Fri Jan 11 12:13:25 2002
whisper11, 4, 5, Fri Jan 11 12:13:28 2002
whisper11, 4, 1, Fri Jan 11 12:13:30 2002
whisper11, 4, 4, 0,4,3,5,1,0, images/c9.gif, Fri Jan 11 12:13:30 2002
whisper11, 5, 1, Fri Jan 11 12:13:43 2002
whispe

In [98]:
!wc -l cardsequence/cardS02/cardS020129a.dat
!wc -l cardsequence/cardS02/cardS020129b.dat
!wc -l cardsequence/cardS02/cardS020130a.dat
!wc -l cardsequence/cardS02/cardS020130b.dat
!wc -l cardsequence/cardS02/cardS020131a.dat
!wc -l cardsequence/cardS02/cardS020131b.dat

16254 cardsequence/cardS02/cardS020129a.dat
1237 cardsequence/cardS02/cardS020129b.dat
5386 cardsequence/cardS02/cardS020130a.dat
15505 cardsequence/cardS02/cardS020130b.dat
3667 cardsequence/cardS02/cardS020131a.dat
15624 cardsequence/cardS02/cardS020131b.dat


In [99]:
!head cardsequence/cardS02/cardS020130a.dat
!tail cardsequence/cardS02/cardS020130a.dat

sasseebaby82878, 10, 3, Wed Jan 30 00:00:02 2002
sasseebaby82878, 10, 3, 0,2,1,3,0,0, images/c5.jpg, Wed Jan 30 00:00:02 2002
sasseebaby82878, 11, 3, Wed Jan 30 00:00:08 2002
sasseebaby82878, 11, 1, 0,3,0,0,0,0, images/c9.jpg, Wed Jan 30 00:00:08 2002
sasseebaby82878, 12, 3, Wed Jan 30 00:00:14 2002
sasseebaby82878, 12, 1, 0,3,0,0,0,0, images/ok.jpg, Wed Jan 30 00:00:14 2002
sasseebaby82878, 13, 1, Wed Jan 30 00:00:20 2002
sasseebaby82878, 13, 3, Wed Jan 30 00:00:23 2002
sasseebaby82878, 13, 2, Wed Jan 30 00:00:26 2002
sasseebaby82878, 13, 3, 0,1,3,2,0,0, images/c2.gif, Wed Jan 30 00:00:26 2002
aero, 23, 1, Wed Jan 30 21:26:08 2002
aero, 23, 1, 0,1,0,0,0,0, images/c1.gif, Wed Jan 30 21:26:08 2002
aero, 24, 1, Wed Jan 30 21:26:13 2002
aero, 24, 2, Wed Jan 30 21:26:15 2002
aero, 24, 3, Wed Jan 30 21:26:17 2002
aero, 24, 4, Wed Jan 30 21:26:19 2002
aero, 24, 4, 0,1,2,3,4,0, images/c6.gif, Wed Jan 30 21:26:19 2002
aero, 25, 5, Wed Jan 30 21:26:24 2002
aero, 25, 4, Wed Jan 30 21:26:26 2002


In [100]:
!head cardsequence/cardS02/cardS020130b.dat
!tail cardsequence/cardS02/cardS020130b.dat

test6, 1, 1, Wed Jan 30 00:06:01 2002
test6, 1, 2, Wed Jan 30 00:06:03 2002
test6, 1, 3, Wed Jan 30 00:06:09 2002
test6, 1, 4, Wed Jan 30 00:06:10 2002
test6, 1, 4, 0,1,2,3,4,0, ../../bi/images4/ok.jpg, Wed Jan 30 00:06:10 2002
pineconelv, 1, 2, Wed Jan 30 02:17:34 2002
pineconelv, 1, 1, Wed Jan 30 02:17:43 2002
pineconelv, 1, 3, Wed Jan 30 02:17:44 2002
pineconelv, 1, 5, Wed Jan 30 02:17:46 2002
pineconelv, 1, 4, Wed Jan 30 02:17:47 2002
cj belt, 23, 3, Wed Jan 30 23:57:34 2002
cj belt, 23, 4, Wed Jan 30 23:57:35 2002
cj belt, 23, 5, 0,2,5,1,3,4, ../../bi/images4/c8.jpg, Wed Jan 30 23:57:35 2002
cj belt, 24, 3, Wed Jan 30 23:58:03 2002
cj belt, 24, 1, Wed Jan 30 23:58:16 2002
cj belt, 24, 2, 0,3,1,0,0,0, ../../bi/images4/c3.jpg, Wed Jan 30 23:58:16 2002
cj belt, 25, 3, Wed Jan 30 23:58:32 2002
cj belt, 25, 2, Wed Jan 30 23:58:49 2002
cj belt, 25, 4, Wed Jan 30 23:58:50 2002
cj belt, 25, 3, 0,3,2,4,0,0, ../../bi/images4/c8.gif, Wed Jan 30 23:58:50 2002


In [101]:
!head cardsequence/cardS02/cardS020131a.dat
!tail cardsequence/cardS02/cardS020131a.dat

are1, 1, 2, Thu Jan 31 00:16:58 2002
are1, 1, 3, Thu Jan 31 00:17:09 2002
are1, 1, 5, Thu Jan 31 00:17:12 2002
are1, 1, 3, 0,2,3,5,0,0, images/c2.gif, Thu Jan 31 00:17:12 2002
are1, 2, 5, Thu Jan 31 00:17:28 2002
are1, 2, 2, Thu Jan 31 00:17:31 2002
are1, 2, 3, Thu Jan 31 00:17:33 2002
are1, 2, 4, Thu Jan 31 00:17:35 2002
are1, 2, 4, 0,5,2,3,4,0, images/c5.gif, Thu Jan 31 00:17:35 2002
are1, 3, 5, Thu Jan 31 00:18:05 2002
grimanesa, 23, 1, 0,3,0,0,0,0, images/c9.jpg, Thu Jan 31 22:28:30 2002
grimanesa, 24, 3, Thu Jan 31 22:28:41 2002
grimanesa, 24, 2, Thu Jan 31 22:28:44 2002
grimanesa, 24, 2, 0,3,2,0,0,0, images/c1.gif, Thu Jan 31 22:28:44 2002
grimanesa, 25, 1, Thu Jan 31 22:28:53 2002
grimanesa, 25, 3, Thu Jan 31 22:28:56 2002
grimanesa, 25, 2, Thu Jan 31 22:28:58 2002
grimanesa, 25, 4, Thu Jan 31 22:29:00 2002
grimanesa, 25, 5, Thu Jan 31 22:29:02 2002
grimanesa, 25, 5, 0,1,3,2,4,5, images/c1.gif, Thu Jan 31 22:29:02 2002


In [102]:
!head cardsequence/cardS02/cardS020131b.dat
!tail cardsequence/cardS02/cardS020131b.dat

chenrae, 1, 2, Thu Jan 31 00:14:18 2002
chenrae, 1, 4, Thu Jan 31 00:14:20 2002
chenrae, 1, 2, 0,2,4,0,0,0, ../../bi/images4/ok.jpg, Thu Jan 31 00:14:20 2002
chenrae, 2, 3, Thu Jan 31 00:14:24 2002
chenrae, 2, 1, 0,3,0,0,0,0, ../../bi/images4/c2.jpg, Thu Jan 31 00:14:24 2002
chenrae, 3, 3, Thu Jan 31 00:14:28 2002
chenrae, 3, 4, Thu Jan 31 00:14:29 2002
chenrae, 3, 2, Thu Jan 31 00:14:31 2002
chenrae, 3, 3, 0,3,4,2,0,0, ../../bi/images4/c10.jpg, Thu Jan 31 00:14:31 2002
chenrae, 4, 2, Thu Jan 31 00:14:37 2002
twang, 23, 1, Thu Jan 31 23:56:47 2002
twang, 23, 4, Thu Jan 31 23:56:55 2002
twang, 23, 5, Thu Jan 31 23:56:57 2002
twang, 23, 3, Thu Jan 31 23:56:58 2002
twang, 23, 2, Thu Jan 31 23:57:00 2002
twang, 23, 5, 0,1,4,5,3,2, ../../bi/images4/c5.jpg, Thu Jan 31 23:57:00 2002
twang, 24, 5, Thu Jan 31 23:57:04 2002
twang, 24, 1, 0,5,0,0,0,0, ../../bi/images4/c7.gif, Thu Jan 31 23:57:04 2002
twang, 25, 2, Thu Jan 31 23:57:09 2002
twang, 25, 1, 0,2,0,0,0,0, ../../bi/images4/c3.gif, Thu Ja

In [76]:
!cat cardsequence/cardS02/cardS020129a.dat | head
!cat cardsequence/cardS02/cardS020129a.dat | tail

tge, 1, 4, Tue Jan 29 00:28:30 2002
tge, 1, 3, Tue Jan 29 00:28:41 2002
tge, 1, 5, Tue Jan 29 00:28:45 2002
tge, 1, 3, 0,4,3,5,0,0, images/ok.jpg, Tue Jan 29 00:28:45 2002
tge, 2, 1, Tue Jan 29 00:28:58 2002
tge, 2, 2, Tue Jan 29 00:29:03 2002
tge, 2, 4, Tue Jan 29 00:29:08 2002
tge, 2, 3, 0,1,2,4,0,0, images/ok.jpg, Tue Jan 29 00:29:08 2002
tge, 3, 3, Tue Jan 29 00:29:19 2002
tge, 3, 5, Tue Jan 29 00:29:22 2002
sasseebaby82878, 7, 1, 0,2,0,0,0,0, images/c3.gif, Tue Jan 29 23:59:27 2002
sasseebaby82878, 8, 2, Tue Jan 29 23:59:34 2002
sasseebaby82878, 8, 1, Tue Jan 29 23:59:37 2002
sasseebaby82878, 8, 2, 0,2,1,0,0,0, images/c4.gif, Tue Jan 29 23:59:37 2002
sasseebaby82878, 9, 2, Tue Jan 29 23:59:43 2002
sasseebaby82878, 9, 1, Tue Jan 29 23:59:46 2002
sasseebaby82878, 9, 3, Tue Jan 29 23:59:48 2002
sasseebaby82878, 9, 3, 0,2,1,3,0,0, images/ok.jpg, Tue Jan 29 23:59:48 2002
sasseebaby82878, 10, 2, Tue Jan 29 23:59:54 2002
sasseebaby82878, 10, 1, Tue Jan 29 23:59:58 2002


In [77]:
!cat cardsequence/cardS02/cardS020129b.dat | head
!cat cardsequence/cardS02/cardS020129b.dat | tail

test4, 1, 1, Mon Jan 28 13:48:13 2002
test4, 1, 1, 0,1,0,0,0,0, ../../bi/images4/c4.gif, Mon Jan 28 13:48:13 2002
test4, 2, 2, Mon Jan 28 13:48:18 2002
test4, 2, 3, Mon Jan 28 13:48:22 2002
test4, 2, 4, Mon Jan 28 13:48:25 2002
test4, 2, 5, Mon Jan 28 13:48:27 2002
test4, 2, 4, 0,2,3,4,5,0, ../../bi/images4/c3.jpg, Mon Jan 28 13:48:27 2002
test4, 3, 2, Mon Jan 28 13:48:36 2002
test4, 3, 1, Mon Jan 28 13:48:38 2002
test4, 3, 3, Mon Jan 28 13:48:40 2002
yellow, 24, 1, Tue Jan 29 23:24:47 2002
yellow, 24, 2, Tue Jan 29 23:24:48 2002
yellow, 24, 4, Tue Jan 29 23:24:50 2002
yellow, 24, 5, Tue Jan 29 23:24:51 2002
yellow, 24, 5, 0,3,1,2,4,5, ../../bi/images4/ok.jpg, Tue Jan 29 23:24:51 2002
yellow, 25, 3, Tue Jan 29 23:24:54 2002
yellow, 25, 5, Tue Jan 29 23:24:55 2002
yellow, 25, 4, Tue Jan 29 23:24:56 2002
yellow, 25, 2, Tue Jan 29 23:24:57 2002
yellow, 25, 4, 0,3,5,4,2,0, ../../bi/images4/c4.gif, Tue Jan 29 23:24:57 2002


In [80]:
!wc -l cardsequence/cardS02/cardS020129a.dat

16254 cardsequence/cardS02/cardS020129a.dat


In [81]:
!wc -l cardsequence/cardS02/cardS020129b.dat

1237 cardsequence/cardS02/cardS020129b.dat


In [65]:
#from gotpsi_sequentialcard.utils import parse_trial_file

parsed_trials = parse_trial_file("/content/cardsequence/cardS02/cardS020202.dat")

In [42]:
max([x.trial for x in parsed_trials])

25

In [44]:
parsed_trials2 = parse_trial_file("/content/cardsequence/cardS18/cardS180101.dat")

In [46]:
max([x.trial for x in parsed_trials2])

25

In [52]:
[x for x in parsed_trials2 if x.userid=="ovalpool"]

[Trial(userid='ovalpool', trial=1, target_card=3, click_sequence=[3], image='c6.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 44)),
 Trial(userid='ovalpool', trial=2, target_card=2, click_sequence=[4, 3, 2], image='c8.gif', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 48)),
 Trial(userid='ovalpool', trial=3, target_card=4, click_sequence=[4], image='c9.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 50)),
 Trial(userid='ovalpool', trial=4, target_card=3, click_sequence=[4, 5, 3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 16, 54)),
 Trial(userid='ovalpool', trial=5, target_card=4, click_sequence=[5, 3, 2, 1, 4], image='c7.gif', timestamp=datetime.datetime(2018, 1, 1, 2, 17)),
 Trial(userid='ovalpool', trial=6, target_card=3, click_sequence=[3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 17, 3)),
 Trial(userid='ovalpool', trial=7, target_card=1, click_sequence=[5, 3, 2, 1], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 2, 17, 8)),


In [49]:
[x for x in parsed_trials2 if x.userid=="rada"]

[Trial(userid='rada', trial=1, target_card=3, click_sequence=[1, 3], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 7)),
 Trial(userid='rada', trial=2, target_card=2, click_sequence=[1, 4, 2], image='c5.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 13)),
 Trial(userid='rada', trial=3, target_card=2, click_sequence=[1, 3, 5, 2], image='c6.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 20)),
 Trial(userid='rada', trial=4, target_card=5, click_sequence=[2, 4, 5], image='ok.jpg', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 25)),
 Trial(userid='rada', trial=5, target_card=5, click_sequence=[2, 1, 4, 3, 5], image='c6.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 35)),
 Trial(userid='rada', trial=6, target_card=4, click_sequence=[5, 2, 3, 1, 4], image='c9.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 41)),
 Trial(userid='rada', trial=7, target_card=2, click_sequence=[4, 2], image='c4.gif', timestamp=datetime.datetime(2018, 1, 1, 1, 7, 45)),
 Trial(useri

In [39]:
!cat cardsequence/cardS02/cardS020202.dat | grep 'chrisreed'

chrisreed, 1, 2, Sat Feb  2 16:45:36 2002
chrisreed, 1, 3, Sat Feb  2 16:45:40 2002
chrisreed, 1, 2, 0,2,3,0,0,0, ../../bi/images4/c5.gif, Sat Feb  2 16:45:40 2002
chrisreed, 2, 2, Sat Feb  2 16:45:46 2002
chrisreed, 2, 4, Sat Feb  2 16:45:48 2002
chrisreed, 2, 3, Sat Feb  2 16:45:49 2002
chrisreed, 2, 3, 0,2,4,3,0,0, ../../bi/images4/c3.gif, Sat Feb  2 16:45:49 2002
chrisreed, 3, 5, Sat Feb  2 16:45:53 2002
chrisreed, 3, 1, 0,5,0,0,0,0, ../../bi/images4/c3.gif, Sat Feb  2 16:45:53 2002
chrisreed, 4, 2, Sat Feb  2 16:45:56 2002
chrisreed, 4, 1, 0,2,0,0,0,0, ../../bi/images4/ok.jpg, Sat Feb  2 16:45:56 2002
chrisreed, 5, 4, Sat Feb  2 16:46:05 2002
chrisreed, 5, 3, Sat Feb  2 16:46:07 2002
chrisreed, 5, 1, Sat Feb  2 16:46:08 2002
chrisreed, 5, 3, 0,4,3,1,0,0, ../../bi/images4/c3.jpg, Sat Feb  2 16:46:08 2002
chrisreed, 6, 2, Sat Feb  2 16:46:13 2002
chrisreed, 6, 5, Sat Feb  2 16:46:14 2002
chrisreed, 6, 2, 0,2,5,0,0,0, ../../bi/images4/ok.jpg, Sat Feb  2 16:46:14 2002
chrisreed, 7, 4,